In [ ]:
from neo4j import GraphDatabase
import pandas as pd

URI = "neo4j://127.0.0.1:7687" 
USERNAME = "neo4j"
PASSWORD = "12345678"
TARGET_DATABASE = "medgraphanalytics"

print(f"Connecting to Neo4j Database: '{TARGET_DATABASE}'...")
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def run_gds_query(query):
    with driver.session(database=TARGET_DATABASE) as session:
        return session.run(query).data()

try:
    print("Projecting graph into memory for high-speed algorithmic processing...")
    
    run_gds_query("CALL gds.graph.drop('hetionet_graph', false)")
    
    run_gds_query("""
    CALL gds.graph.project(
      'hetionet_graph',
      'Entity',
      {
        RELATES_TO: {
          orientation: 'UNDIRECTED'
        }
      }
    )
    """)

    print("Executing PageRank, Betweenness, Eigenvector, Louvain, LPA, and Clustering...")
    query_metrics = """
    CALL gds.pageRank.stream('hetionet_graph') YIELD nodeId, score AS pagerank
    WITH gds.util.asNode(nodeId).id AS id, pagerank

    CALL gds.betweenness.stream('hetionet_graph') YIELD nodeId, score AS betweenness
    WITH id, pagerank, betweenness WHERE gds.util.asNode(nodeId).id = id

    CALL gds.eigenvector.stream('hetionet_graph') YIELD nodeId, score AS eigenvector
    WITH id, pagerank, betweenness, eigenvector WHERE gds.util.asNode(nodeId).id = id

    CALL gds.louvain.stream('hetionet_graph') YIELD nodeId, communityId AS louvain
    WITH id, pagerank, betweenness, eigenvector, louvain WHERE gds.util.asNode(nodeId).id = id

    CALL gds.labelPropagation.stream('hetionet_graph') YIELD nodeId, communityId AS lpa
    WITH id, pagerank, betweenness, eigenvector, louvain, lpa WHERE gds.util.asNode(nodeId).id = id

    CALL gds.localClusteringCoefficient.stream('hetionet_graph') YIELD nodeId, localClusteringCoefficient AS clustering
    RETURN id, pagerank, betweenness, eigenvector, louvain, lpa, clustering
    """

    results = run_gds_query(query_metrics)

    print("Writing structural features to graph_metrics.csv...")
    metrics_df = pd.DataFrame(results)
    metrics_df.to_csv('graph_metrics.csv', index=False)
    
    print(f"Pipeline complete! Extracted complex features for {len(metrics_df):,} nodes.")

except Exception as e:
    print(f"An error occurred during GDS execution:\n{e}")

finally:
    try:
        run_gds_query("CALL gds.graph.drop('hetionet_graph', false)")
    except:
        pass
    driver.close()
    print("Memory cleared and driver safely closed.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('hetionet_graph', false)"


Connecting to Neo4j Database: 'medgraphanalytics'...
Projecting graph into memory for high-speed algorithmic processing...
Executing PageRank, Betweenness, Eigenvector, Louvain, LPA, and Clustering...
